# v7 — Three-class int8 comparison (prior evaluation pending)
Reuses the saved v1 model. The current evaluator compares fp32 CPU with int8 CPU on the same
balanced-test subset and exports quantized weights. It does not yet score quantized natural/prior
variants; those remain pending. Model loading falls back to the v1 training recipe only if no saved model is available.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/btp_v7_int8_3class_prior'
    V1_ZIP = '/content/drive/MyDrive/btp_v1_roberta_base_3class/btp_v1_roberta_base_3class.zip'
except Exception as e:
    print(f'Drive mount failed ({e}); the v1 model will be retrained.')
    DRIVE, V1_ZIP = '/content/btp_v7_int8_3class_prior', ''
!mkdir -p {DRIVE}
!test -d /content/BTP || git clone -q https://github.com/Abhijeet-SP/BTP.git /content/BTP
%cd /content/BTP
!git pull --ff-only && git log --oneline -1
RESULTS = 'before_mid_evaluation/experiments/v7_int8_3class_prior/results'
!pip -q install -r requirements.txt

In [ ]:
import os
# Re-set here: a Colab restart (e.g. after the pip upgrade) wipes variables and the working dir
%cd /content/BTP
RESULTS = 'before_mid_evaluation/experiments/v7_int8_3class_prior/results'
DRIVE = '/content/drive/MyDrive/btp_v7_int8_3class_prior' if os.path.exists('/content/drive/MyDrive') else '/content/btp_v7_int8_3class_prior'
V1_ZIP = '/content/drive/MyDrive/btp_v1_roberta_base_3class/btp_v1_roberta_base_3class.zip'
# ~8 min: same balanced test_balanced.csv + test_natural.csv as v1 (seeded)
!python prepare_data.py

In [ ]:
import os
# Re-set here: a Colab restart (e.g. after the pip upgrade) wipes variables and the working dir
%cd /content/BTP
RESULTS = 'before_mid_evaluation/experiments/v7_int8_3class_prior/results'
DRIVE = '/content/drive/MyDrive/btp_v7_int8_3class_prior' if os.path.exists('/content/drive/MyDrive') else '/content/btp_v7_int8_3class_prior'
V1_ZIP = '/content/drive/MyDrive/btp_v1_roberta_base_3class/btp_v1_roberta_base_3class.zip'
# The v1 model: from the zip the v1 notebook left on Drive, else retrain it with the v1 recipe (~42 min)
if not os.path.exists('models/roberta_base_3class/config.json'):
    if V1_ZIP and os.path.exists(V1_ZIP):
        !unzip -qo "{V1_ZIP}" 'models/roberta_base_3class/*'
    else:
        print('v1 zip not on Drive, retraining v1')
        !python finetune_roberta.py --name roberta_base_3class --batch-size 32 --results-dir {RESULTS} --ckpt-dir {DRIVE}/ckpt
assert os.path.exists('models/roberta_base_3class/config.json'), 'No v1 model: scroll up. Re-run the cell to resume.'
print('v1 model ready')

In [ ]:
import os
# Re-set here: a Colab restart (e.g. after the pip upgrade) wipes variables and the working dir
%cd /content/BTP
RESULTS = 'before_mid_evaluation/experiments/v7_int8_3class_prior/results'
DRIVE = '/content/drive/MyDrive/btp_v7_int8_3class_prior' if os.path.exists('/content/drive/MyDrive') else '/content/btp_v7_int8_3class_prior'
V1_ZIP = '/content/drive/MyDrive/btp_v1_roberta_base_3class/btp_v1_roberta_base_3class.zip'
# Matched CPU comparison on 4,000 balanced-test rows; natural/prior quantized scoring remains pending
!python evaluate_model.py --model models/roberta_base_3class --quantize --results-dir {RESULTS} 2>&1 | tee /tmp/quantize.log
!mkdir -p {RESULTS}/logs && cp /tmp/quantize.log {RESULTS}/logs/quantize.log

In [ ]:
import os
# Re-set here: a Colab restart (e.g. after the pip upgrade) wipes variables and the working dir
%cd /content/BTP
RESULTS = 'before_mid_evaluation/experiments/v7_int8_3class_prior/results'
DRIVE = '/content/drive/MyDrive/btp_v7_int8_3class_prior' if os.path.exists('/content/drive/MyDrive') else '/content/btp_v7_int8_3class_prior'
V1_ZIP = '/content/drive/MyDrive/btp_v1_roberta_base_3class/btp_v1_roberta_base_3class.zip'
import json
m = json.load(open(f'{RESULTS}/metrics.json'))
comparison = m['roberta_base_3class_quantized']
for variant in ('fp32_cpu', 'int8_cpu'):
    scores = comparison[variant]
    print(f"{variant:12s} acc {scores['accuracy']:.4f}  macro-F1 {scores['macro_f1']:.4f}  "
          f"{scores['reviews_per_second']} reviews/s")
print(f"Stored source weights: {comparison['fp32_size_mb']} MB; int8 export: {comparison['int8_size_mb']} MB")
print('Quantized natural/prior evaluation is pending; this run scores the balanced subset only.')


In [ ]:
import os
# Re-set here: a Colab restart (e.g. after the pip upgrade) wipes variables and the working dir
%cd /content/BTP
RESULTS = 'before_mid_evaluation/experiments/v7_int8_3class_prior/results'
DRIVE = '/content/drive/MyDrive/btp_v7_int8_3class_prior' if os.path.exists('/content/drive/MyDrive') else '/content/btp_v7_int8_3class_prior'
V1_ZIP = '/content/drive/MyDrive/btp_v1_roberta_base_3class/btp_v1_roberta_base_3class.zip'
!zip -qr btp_v7_int8_3class_prior.zip models/roberta_base_3class_int8.pt before_mid_evaluation/experiments/v7_int8_3class_prior/results && ls -lh btp_v7_int8_3class_prior.zip
!cp btp_v7_int8_3class_prior.zip {DRIVE}/
from google.colab import files
files.download('btp_v7_int8_3class_prior.zip')